# Ground state of the 2D Fermi-Hubbard model benchmark

This notebook benchmarks quantum phase estimation (QPE) for the ground state of the
**Fermi-Hubbard model on a two-dimensional square lattice**, and estimates the fault-tolerant
resources required to run it across a sweep of lattice sizes.

$$
H = -T\sum_{\langle i,j\rangle,\;\sigma\in\{\uparrow,\downarrow\}} \left(c^{\dagger}_{i\sigma}c_{j\sigma} + \text{h.c.}\right)
  \; + \; U\sum_{i} n_{i\uparrow}\,n_{i\downarrow},
\qquad n_{i\sigma}=c^{\dagger}_{i\sigma}c_{i\sigma}
$$

Here $c^{\dagger}_{i\sigma}$ and $c_{j\sigma}$ are the fermionic creation and annihilation operators on lattice sites $i,j$, and $n_{i\sigma}$ is the on-site occupation with spin $\sigma$.

**Benchmark specification**

| Quantity | Value |
|---|---|
| Lattice | 2D square, periodic in both directions, $N=L^{2}$ sites |
| Parameters | $U/T = 1/8$, $f = 0.875$, i.e. $2fN$ electrons |
| Target | Ground-state energy to $0.0051\,T$ per site |
| Hardware model | Majorana qubits, measurement error rate $10^{-5}$ |

**References**

- Childs, Andrew M., et al. "Theory of Trotter error with commutator scaling." *Physical Review X* **11**.1 (2021): 011020. [arXiv:1912.08854](https://arxiv.org/abs/1912.08854)
- Kivlichan, Ian D., et al. "Improved fault-tolerant quantum simulation of condensed-phase correlated electrons via Trotterization." *Quantum* **4** (2020): 296. [arXiv:1902.10673](https://arxiv.org/abs/1902.10673)
- Campbell, Earl T. "Early fault-tolerant simulations of the Hubbard model." *Quantum Science & Technology* **7**.1 (2022): 015007. [arXiv:2012.09238](https://arxiv.org/abs/2012.09238)
- Schubert, Christoph, and Christian B. Mendl. "Trotter error with commutator scaling for the Fermi-Hubbard model." *Physical Review B* **108** (2023): 195105. [arXiv:2306.10603](https://arxiv.org/abs/2306.10603)
- Bärtschi, Andreas, et al. "Potential applications of quantum computing at Los Alamos National Laboratory." (2024), Chapter 5, Application 1. [arXiv:2406.06625](https://arxiv.org/abs/2406.06625)
- Nielsen, Michael A., and Isaac L. Chuang. *Quantum Computation and Quantum Information*, Sec. 5.2.1.

**Requirements**

```bash
pip install 'qdk-chemistry[jupyter,qre]'
```

In [ ]:
import signal
from contextlib import contextmanager

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    Circuit,
    LatticeGraph,
    MajoranaMapping,
)
from qdk_chemistry.data.circuit import QsharpFactoryData
from qdk_chemistry.utils import Logger
from qdk_chemistry.utils.model_hamiltonians import create_hubbard_hamiltonian
from qdk_chemistry.utils.qsharp import (
    QSHARP_UTILS,
    create_qsharp_context,
    use_qsharp_context,
)

Logger.set_global_level(Logger.LogLevel.off)

HOPPING_T = 1.0                                          # T > 0, energies are quoted in units of T
U_OVER_T = 1.0 / 8.0                                     # on-site repulsion / hopping
COULOMB_U = U_OVER_T * HOPPING_T
FILLING = 0.875                                          # 2 * FILLING * N electrons in total
TARGET_PRECISION_PER_SITE = 0.0051   # energy accuracy per lattice site, in units of T
BENCHMARK_LATTICE_SIZES = list(range(2, 11, 2)) + list(range(20, 201, 10))
STEP_TIMEOUT_SECONDS = 600           # ceiling on circuit construction and on estimation
EXAMPLE_NUM_SITES = 50

# QDK interpreters are thread-affine, so this context belongs to the notebook execution thread.
QSHARP_CONTEXT = create_qsharp_context()

_SUPPORTS_ALARM = hasattr(signal, "SIGALRM") and hasattr(signal, "setitimer")
@contextmanager
def time_limit(seconds: int, what: str):
    """Abort *what* after *seconds* on platforms that provide SIGALRM."""
    if not _SUPPORTS_ALARM:
        yield
        return

    def _timed_out(signum, frame):
        raise TimeoutError(f"{what} exceeded {seconds} s")

    previous = signal.signal(signal.SIGALRM, _timed_out)
    signal.setitimer(signal.ITIMER_REAL, seconds)
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, previous)


def target_precision(size: int) -> float:
    """Absolute ground-state energy accuracy required of an L x L lattice."""
    return TARGET_PRECISION_PER_SITE * size * size


print(f"Hubbard model: U/T = {U_OVER_T:g}  (T = {HOPPING_T:g}, U = {COULOMB_U:g}) Filling factor f = {FILLING:g}")
print(f"Benchmark lattice sizes L = {BENCHMARK_LATTICE_SIZES}")

## The lattice and the qubit Hamiltonian

`LatticeGraph.square` builds the $L\times L$ lattice and `create_hubbard_hamiltonian` turns it into a
fermionic Hamiltonian. The Jordan-Wigner mapping then produces the qubit Hamiltonian on $2N$ qubits
(one per spin-orbital).

The benchmark asks for $2fN$ electrons at $f = 0.875$, which is not an integer for every $N$. We round to the
nearest electron count and split it as evenly as possible between the two spin sectors, so $|S_z| \le 1/2$.

The mapping lays the spin-orbitals out in blocks: qubits $0 \dots N-1$ carry the spin-up sites and qubits
$N \dots 2N-1$ the spin-down sites, which is why the on-site $U$ term shows up as a $Z_i Z_{i+N}$ coupling.

In [ ]:
def qubit_operator(size: int):
    """Jordan-Wigner qubit Hamiltonian of the periodic size x size Hubbard lattice."""
    num_sites = size * size
    lattice = LatticeGraph.square(size, size, periodic_x=True, periodic_y=True)
    hamiltonian = create_hubbard_hamiltonian(
        lattice, epsilon=0.0, t=HOPPING_T, U=COULOMB_U
    )
    operator = create("qubit_mapper").run(
        hamiltonian, mapping=MajoranaMapping.jordan_wigner(2 * num_sites)
    )
    return operator


def num_electrons(size: int) -> int:
    """Electron count nearest to the requested filling 2 * f * N."""
    return round(2 * FILLING * size * size)


example = qubit_operator(EXAMPLE_NUM_SITES)
print(f"{EXAMPLE_NUM_SITES}x{EXAMPLE_NUM_SITES} lattice: {example.num_qubits} qubits, {len(example.pauli_strings)} Pauli terms, "
      f"{num_electrons(EXAMPLE_NUM_SITES)} electrons, lambda = {example.schatten_norm:g}")

<a id="sizing"></a>

## Sizing the QPE and Trotter parameters

Both algorithm knobs follow from one input, the target precision $\epsilon(L)$, and one property of the
qubit Hamiltonian $H = \sum_j c_j P_j$: its coefficient 1-norm

$$
\lambda = \sum_j |c_j|,
$$

returned by `QubitOperator.schatten_norm`.

**Evolution time.** Every eigenvalue of $H$ lies in $[-\lambda, \lambda]$, so

$$
t_0 = \frac{\pi}{\lambda}
$$

maps the spectrum of $H t_0$ into $[-\pi, \pi]$ and the QPE phase into $[-\tfrac{1}{2}, \tfrac{1}{2})$. The
whole phase register is used and no eigenvalue aliases.

**Error budget.** The target precision is split between the two independent error sources,
$\epsilon = \epsilon_{\mathrm{QPE}} + \epsilon_{\mathrm{Trotter}}$.

**Phase register width.** An $m$-bit register resolves the phase to $2^{-m}$, hence the energy to
$2\lambda / 2^{m}$. Resolving $\epsilon_{\mathrm{QPE}}$ therefore costs
$\lceil \log_2 (2\lambda/\epsilon_{\mathrm{QPE}}) \rceil$ bits, and reading those leading bits correctly with
probability $1 - \delta$ costs $\lceil \log_2 (2 + 1/2\delta) \rceil$ additional guard bits (Nielsen & Chuang,
Sec. 5.2.1):

$$
m = \left\lceil \log_2 \frac{2\lambda}{\epsilon_{\mathrm{QPE}}} \right\rceil
  + \left\lceil \log_2 \left( 2 + \frac{1}{2\delta} \right) \right\rceil .
$$

**Trotter steps.** QPE applies powers of a *single* fixed unitary $U = S_2(t_0)$. A product formula realizes
$U$ exactly as $e^{-i H_{\mathrm{eff}} t_0}$ for some effective Hamiltonian, so the readout is an exact
eigenvalue of $H_{\mathrm{eff}}$ and the Trotter error does not accumulate over the $2^{m}-1$ repetitions --
it shows up once, as the bias
$\lVert H_{\mathrm{eff}} - H \rVert \le \lVert S_2(t_0) - e^{-iHt_0} \rVert / t_0$. Holding that bias below
$\epsilon_{\mathrm{Trotter}}$ means the product formula must be accurate to

$$
\delta_{\mathrm{Trotter}} = \epsilon_{\mathrm{Trotter}} \, t_0
$$

in spectral norm. Childs *et al.* Prop. 10 bounds the error of one Strang step by
$\tfrac{t^3}{12}(\dots) + \tfrac{t^3}{24}(\dots)$; distributing the norms term-by-term over the Pauli terms
gives the quantity that `qdk_chemistry.utils.pauli_commutation.commutator_bound_second_order` evaluates,

$$
\alpha_2 = \!\!\sum_{k>j,\,l>j}\!\! \bigl\lVert [c_l P_l, [c_k P_k, c_j P_j]] \bigr\rVert
         + \tfrac{1}{2} \sum_{k>j} \bigl\lVert [c_j P_j, [c_j P_j, c_k P_k]] \bigr\rVert ,
\qquad
r = \left\lceil \sqrt{\frac{\alpha_2}{12}} \; \frac{t_0^{3/2}}{\sqrt{\delta_{\mathrm{Trotter}}}} \right\rceil .
$$

In Kivlichan *et al.*'s notation this is exactly $\alpha_2 = 12\,W$, with $W$ their "Trotter error norm"
(their Eq. 6).

**Allocating the budget.** Campbell (App. F) shows the split is not free -- there is an optimum. Write the
base unitary as $U = S_2(\Delta t)^{r}$ over total time $t_0 = r\,\Delta t$, applied $2^{m}$ times by the
ladder. The energy bias depends only on the *step* time, $\epsilon_{\mathrm{Trotter}} = W \Delta t^{2}$,
while the readout resolution is $\epsilon_{\mathrm{QPE}} = 2\pi/(t_0 2^{m})$. The total cost is

$$
2^{m} \cdot r \;=\; \frac{2\pi}{t_0\,\epsilon_{\mathrm{QPE}}}\cdot\frac{t_0}{\Delta t}
\;=\; \frac{2\pi}{\Delta t \; \epsilon_{\mathrm{QPE}}},
$$

which is independent of $t_0$ and $r$ *separately* -- only the step time matters. Substituting
$\Delta t = \sqrt{\epsilon_{\mathrm{Trotter}}/W}$ and maximizing
$\sqrt{\epsilon_{\mathrm{Trotter}}}\,(\epsilon - \epsilon_{\mathrm{Trotter}})$ gives

$$
\epsilon_{\mathrm{Trotter}} = \tfrac{1}{3}\epsilon, \qquad \epsilon_{\mathrm{QPE}} = \tfrac{2}{3}\epsilon,
$$

exactly Campbell's Eq. (116) ("$\Delta_{TS} = \delta/3$ and $\Delta_{PE} = 2\delta/3$"), here rederived for a
textbook $m$-bit ladder rather than his single-ancilla cost model.

That optimum assumes $m$ is continuous. Because $m$ is an integer, the resolution $2\lambda/2^{m}$ jumps by
factors of two, and a fixed $1/3$ split usually leaves part of the budget stranded. The planner below instead
enumerates $m$, hands **every** unit of leftover budget to the Trotter term (the largest $\Delta t$, hence the
smallest $r$), and keeps the $m$ that minimizes $2^{m} r$. For this benchmark that is $1.4$–$2\times$ cheaper
than the fixed $50/50$ split and saves one phase bit, which halves the ladder repetitions.

### Evaluating $\alpha_2$ without the cubic Python loop

The reference implementation walks all $O(n_{\mathrm{terms}}^{3})$ triples and tests each nested commutator on
Pauli *strings*, which costs about 105 s already at $L = 6$ and days beyond $L = 10$. The same number can be
obtained from linear algebra.

Write each Pauli term as a symplectic vector $v_j = (x_j \,|\, z_j) \in \mathbb{F}_2^{2n_q}$, so that $P_a$ and
$P_b$ anticommute exactly when $\langle v_a, v_b \rangle = x_a\!\cdot\!z_b + z_a\!\cdot\!x_b = 1 \pmod 2$. Then

- $[c_kP_k, c_jP_j] \neq 0$ iff $\langle v_j, v_k \rangle = 1$, in which case the commutator is
  $2c_jc_k P_kP_j$, whose symplectic vector is $v_j + v_k$;
- so $[c_lP_l, [c_kP_k, c_jP_j]] \neq 0$ iff additionally
  $\langle v_l, v_j + v_k \rangle = \langle v_l, v_j \rangle \oplus \langle v_l, v_k \rangle = 1$, i.e. **$P_l$
  anticommutes with exactly one of $P_j, P_k$**.

Every Pauli-string test collapses into the single anticommutation matrix
$A_{ab} = \langle v_a, v_b \rangle$, and the inner sum over $l$ becomes
$\sum_l w_l (A_{lj} + A_{lk} - 2A_{lj}A_{lk})$ -- a matrix product. The result is *identical* to
`qdk_chemistry.utils.pauli_commutation.commutator_bound_second_order`, not an approximation.


In [ ]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd

TROTTER_ORDER = 2                    # Suzuki-Trotter product-formula order
QPE_FAILURE_PROBABILITY = 0.1        # 1 - confidence that the readout meets the target precision
WEIGHT_THRESHOLD = 1e-12
ALPHA2_EXACT_MAX_TERMS = 4000        # exact bound costs about 1.2e-10 * n_terms**3 seconds
ALPHA2_PER_SITE = 76.3               # extensive plateau; matches the exact value to 0.1% by L = 14
MAX_RESOLUTION_BITS = 64


def _symplectic(labels, num_qubits):
    """Binary x/z matrices: row t, column q holds qubit q of Pauli term t."""
    x = np.zeros((len(labels), num_qubits), dtype=np.float32)
    z = np.zeros((len(labels), num_qubits), dtype=np.float32)
    for t, label in enumerate(labels):
        for q, char in enumerate(reversed(label)):
            if char in ("X", "Y"):
                x[t, q] = 1.0
            if char in ("Z", "Y"):
                z[t, q] = 1.0
    return x, z


def commutator_bound_second_order_fast(operator, weight_threshold=WEIGHT_THRESHOLD):
    """Exact order-2 commutator bound alpha_2, vectorized over the anticommutation matrix."""
    terms = operator.get_real_coefficients(tolerance=weight_threshold)
    labels = [label for label, _ in terms]
    weights = np.abs(np.asarray([coeff for _, coeff in terms], dtype=np.float64))
    count = len(labels)
    x, z = _symplectic(labels, operator.num_qubits)

    # anti[a, b] == 1 exactly when P_a and P_b anticommute
    anti = np.mod(x @ z.T + z @ x.T, 2.0).astype(np.float64)

    # suffix[j, k] = sum over terms l >= j of w_l * anti[l, k]
    suffix = np.cumsum((weights[:, None] * anti)[::-1], axis=0)[::-1]

    term1 = 0.0
    for j in range(count - 1):
        tail_weights = weights[j + 1:]
        column = anti[j + 1:, j]                     # anti[l, j] for l > j
        block = anti[j + 1:, j + 1:]                 # anti[l, k] for l, k > j
        both = (tail_weights * column) @ block       # sum_l w_l anti[l,j] anti[l,k]
        # sum_l w_l * (anti[l,j] XOR anti[l,k]), i.e. P_l anticommutes with exactly one
        exactly_one = float(tail_weights @ column) + suffix[j + 1, j + 1:] - 2.0 * both
        term1 += weights[j] * float((tail_weights * anti[j, j + 1:]) @ exactly_one)

    # [P_j,[P_j,P_k]] != 0 exactly when P_j and P_k anticommute
    term2 = sum(
        weights[j] ** 2 * float((weights[j + 1:] * anti[j, j + 1:]).sum())
        for j in range(count - 1)
    )
    return 4.0 * term1 + 2.0 * term2


def alpha2_bound(operator):
    """Order-2 commutator bound for the operator.

    The exact bound is cubic in the term count and quadratic in memory, so past
    ALPHA2_EXACT_MAX_TERMS it is replaced by the extensive plateau: alpha_2 enters
    only as sqrt(alpha_2), and r has already saturated at 1 by then. The plateau is
    cross-checked against Campbell's closed form further down.
    """
    if len(operator.pauli_strings) <= ALPHA2_EXACT_MAX_TERMS:
        return commutator_bound_second_order_fast(operator)
    return ALPHA2_PER_SITE * (operator.num_qubits // 2)


GUARD_BITS = math.ceil(math.log2(2 + 1 / (2 * QPE_FAILURE_PROBABILITY)))


@dataclass(frozen=True)
class QpeParameters:
    """Algorithm parameters derived from a target precision."""

    one_norm: float        # lambda
    evolution_time: float  # t_0
    num_bits: int          # m, including guard bits
    num_divisions: int     # r
    alpha2: float          # commutator bound
    trotter_budget: float  # epsilon_Trotter actually allocated


def plan_qpe(one_norm: float, precision: float, alpha2: float) -> QpeParameters:
    """Choose (m, r) minimizing the ladder cost 2**m * r for a target precision.

    Implements Campbell's App. F optimization with m constrained to integers: for
    every candidate resolution the leftover budget goes entirely to the Trotter
    term, which maximizes the step time and therefore minimizes r.
    """
    evolution_time = math.pi / one_norm          # largest time with no eigenvalue aliasing
    error_constant = alpha2 / 12.0               # Campbell/Kivlichan W

    best = None
    for resolution_bits in range(1, MAX_RESOLUTION_BITS):
        readout_budget = 2 * one_norm / 2**resolution_bits    # = 2*pi/(t_0 * 2**m)
        if readout_budget >= precision:
            continue                                          # nothing left for Trotter
        trotter_budget = precision - readout_budget
        step_time = math.sqrt(trotter_budget / error_constant)
        divisions = max(1, math.ceil(evolution_time / step_time))
        cost = (2**resolution_bits) * divisions
        if best is None or cost < best[0]:
            best = (cost, resolution_bits, divisions, trotter_budget)

    if best is None:
        raise ValueError(f"no resolution meets precision {precision:g} for lambda {one_norm:g}")

    _, resolution_bits, divisions, trotter_budget = best
    return QpeParameters(
        one_norm=one_norm,
        evolution_time=evolution_time,
        num_bits=resolution_bits + GUARD_BITS,
        num_divisions=divisions,
        alpha2=alpha2,
        trotter_budget=trotter_budget,
    )


def qpe_parameters(operator, precision: float) -> QpeParameters:
    """Derive the QPE register width and Trotter step count for a target precision."""
    return plan_qpe(operator.schatten_norm, precision, alpha2_bound(operator))


parameters = qpe_parameters(example, target_precision(EXAMPLE_NUM_SITES))
print(f"{EXAMPLE_NUM_SITES}x{EXAMPLE_NUM_SITES} lattice, epsilon = {target_precision(EXAMPLE_NUM_SITES):g}: {parameters}")

### Cross-checking the $\alpha_2$ plateau against Campbell's closed form

Past `ALPHA2_EXACT_MAX_TERMS` the planner stops evaluating the exact bound and uses the extensive plateau
$\alpha_2 \approx 76.3\,N$, so for every lattice in this sweep beyond $L \approx 14$ that constant is
load-bearing. Campbell's App. C Thm. 1 gives an *independent, rigorous, $O(1)$* bound to check it against.

For the periodic $L\times L$ lattice his Eqs. (43)–(44) read

$$
W_{SO1} \le \frac{u^{2}}{12}\lVert H_h\rVert + \frac{u\tau^{2}}{12}L^{2}\left(\sqrt5+8\right),
\qquad
W_{SO2} \le \frac{u\tau^{2}}{6}L^{2}\left(\sqrt5+8\right) + \frac{u^{2}}{24}\lVert H_h\rVert,
$$

where $\lVert H_h\rVert$ is the Schatten-1 norm of the *single-spin-species* hopping matrix. That matrix is
free-fermionic, so its eigenvalues are the band energies and the norm is a closed-form momentum sum,

$$
\lVert H_h\rVert = \tau\!\!\sum_{n,m=0}^{L-1}\!\left|2\left(\cos\tfrac{2\pi n}{L}+\cos\tfrac{2\pi m}{L}\right)\right|
\;\xrightarrow[L\to\infty]{}\; \frac{16}{\pi^{2}}\,\tau L^{2}.
$$

The cell below reproduces Campbell's Table 3 and Table 1 as assertions, then compares the three groupings.

**Campbell's number is a cross-check, not a substitute.** His bound assumes the Hamiltonian is split into
$\Gamma = 2$ layers, with the *entire* hopping operator exponentiated at once (via FFFT or Givens rotations).
The `pauli_sequence` mapper used here exponentiates every Pauli term separately, so $\Gamma = n_{\mathrm{terms}}$
and the bound is necessarily larger. The finer the grouping, the larger $\alpha_2$ — which is exactly the
ordering the table shows. Substituting Campbell's value into this circuit would understate $r$ for a circuit
that does not implement his grouping.

In [ ]:
SQRT5_PLUS_8 = math.sqrt(5.0) + 8.0

# Campbell Table 3, ||H_h|| / tau, quoted to 2-3 significant figures.
CAMPBELL_TABLE_3 = {4: 24, 6: 56, 8: 100, 10: 160, 12: 230, 16: 410, 20: 650}
# Campbell Table 1, min(W_SO1, W_SO2) at u/tau = 4.
CAMPBELL_TABLE_1 = {4: 8.7e1, 8: 3.5e2, 16: 1.4e3}


def hopping_trace_norm(size: int, hopping: float = HOPPING_T) -> float:
    """Schatten-1 norm of the single-species hopping matrix on a periodic size x size lattice.

    The hopping operator is free-fermionic, so its eigenvalues are the band
    energies -2*tau*(cos kx + cos ky) over the size**2 allowed momenta.
    """
    momenta = 2.0 * np.pi * np.arange(size) / size
    band = 2.0 * hopping * (np.cos(momenta)[:, None] + np.cos(momenta)[None, :])
    return float(np.abs(band).sum())


def campbell_error_constant(size: int, hopping: float = HOPPING_T, coulomb: float = COULOMB_U) -> float:
    """Campbell App. C Thm. 1 Eqs. (43)-(44): W for the 2-layer split-operator grouping."""
    norm = hopping_trace_norm(size, hopping)
    sites = size * size
    w_so1 = (coulomb**2 / 12.0) * norm + (coulomb * hopping**2 / 12.0) * sites * SQRT5_PLUS_8
    w_so2 = (coulomb * hopping**2 / 6.0) * sites * SQRT5_PLUS_8 + (coulomb**2 / 24.0) * norm
    return min(w_so1, w_so2)


for size, tabulated in CAMPBELL_TABLE_3.items():
    computed = hopping_trace_norm(size, 1.0)
    assert abs(computed - tabulated) / tabulated < 0.02, (size, computed, tabulated)

for size, tabulated in CAMPBELL_TABLE_1.items():
    computed = campbell_error_constant(size, hopping=1.0, coulomb=4.0)
    assert abs(computed - tabulated) / tabulated < 0.02, (size, computed, tabulated)

print(f"Campbell Tables 1 and 3 reproduced; ||H_h||/(tau N) at L=64 = "
      f"{hopping_trace_norm(64, 1.0) / 64**2:.4f} (16/pi^2 = {16 / math.pi**2:.4f})")

# Schubert & Mendl Eq. (44), 3-layer plaquette grouping, per site.
plaquette_per_site = 12.0 * (1.0 / 6.0) * (
    4.4142 * HOPPING_T**3 + 8.0889 * HOPPING_T**2 * COULOMB_U + 1.3062 * HOPPING_T * COULOMB_U**2
)

grouping = pd.DataFrame(
    [
        {
            "L": size,
            "Gamma=2 (Campbell)": 12 * campbell_error_constant(size) / (size * size),
            "Gamma=3 (Schubert-Mendl)": plaquette_per_site,
            "term-by-term (this circuit)": ALPHA2_PER_SITE,
        }
        for size in sorted({4, 8, 16, 20, EXAMPLE_NUM_SITES})
    ]
).set_index("L")
grouping["ratio to Campbell"] = grouping["term-by-term (this circuit)"] / grouping["Gamma=2 (Campbell)"]
grouping

## Reference state and QPE circuit

The benchmark calls for a Hartree-Fock (Fermi-sea) reference. In the site basis that the Jordan-Wigner
mapping works in, the Fermi sea is a *momentum*-space determinant, so preparing it exactly needs a
Givens-rotation network of $O(N^{2})$ two-qubit gates. We use the site-basis occupation-number determinant
with the same particle number and $S_z = 0$ instead, which is a single layer of $X$ gates. The two choices
differ in *overlap* with the ground state -- which sets how many times QPE has to be repeated -- but barely
in *cost*: $O(N^{2})$ gates against the $\sim 10^{7}$ rotations inside the QPE ladder.

`qdk_standard` then assembles the phase register, the controlled $U^{2^{k}}$ ladder and the inverse QFT.
Each controlled power is compiled by the `pauli_sequence` mapper into a Q# operation, so the object handed
to the resource estimator in the next section is the actual circuit.

In [ ]:
IQPE_ITERATION = 0                   # iteration 0 carries the largest power, 2**(m-1)

def reference_state_prep(num_sites: int, electrons: int) -> Circuit:
    """Occupation-number determinant, one X gate per occupied spin-orbital.

    Electrons are split as evenly as possible between the spin-up block
    (qubits 0..N-1) and the spin-down block (qubits N..2N-1).
    """
    num_up = (electrons + 1) // 2
    num_down = electrons // 2
    occupations = (
        [1] * num_up + [0] * (num_sites - num_up)
        + [1] * num_down + [0] * (num_sites - num_down)
    )
    with use_qsharp_context(QSHARP_CONTEXT):
        state_preparation = QSHARP_UTILS.StatePreparation
        params = state_preparation.SingleReferenceParams(bitStrings=occupations, numQubits=2 * num_sites)
        return Circuit(
            qsharp_factory=QsharpFactoryData(
                program=state_preparation.MakeSingleReferenceStateCircuit, parameter=vars(params)
            ),
            qsharp_op=state_preparation.MakePrepareSingleReferenceStateOp(params),
            encoding="jordan-wigner",
        )


def qpe_circuit(operator, parameters: QpeParameters, initial_state: Circuit) -> Circuit:
    """Single IQPE iteration for `operator`, Trotterized according to `parameters`.

    `num_iteration` selects one round instead of the whole ladder, so exactly one
    controlled unitary is compiled rather than `num_bits` of them.
    """
    with use_qsharp_context(QSHARP_CONTEXT):
        builder = create(
            "qpe_circuit_builder",
            "qdk_iterative",
            unitary_builder=AlgorithmRef(
                "hamiltonian_unitary_builder",
                "trotter",
                order=TROTTER_ORDER,
                time=parameters.evolution_time,
                num_divisions=parameters.num_divisions,
            ),
            controlled_circuit_mapper=AlgorithmRef("controlled_circuit_mapper", "pauli_sequence"),
            num_bits=parameters.num_bits,
            num_iteration=IQPE_ITERATION,
        )
        with time_limit(STEP_TIMEOUT_SECONDS, "qpe_circuit"):
            return builder.run(initial_state, operator)[0]


circuit = qpe_circuit(example, parameters, reference_state_prep(EXAMPLE_NUM_SITES * EXAMPLE_NUM_SITES, num_electrons(EXAMPLE_NUM_SITES)))

## Physical resource estimation

The Q# circuit goes straight to `qdk.qre`; the tracer reports the rotation and measurement counts as
compressed `repeat` blocks, so no logical-count formula is needed.

The trace query expands fine-grained rotations into T gates (`PSSPC`) and schedules the logical operations
with lattice surgery (`LatticeSurgery`); `slow_down_factor` trades runtime for fewer magic-state factories.
The ISA query supplies the surface-code instruction (`ThreeAux`) and a generic distillation model
(`RoundBasedFactory`).

The sweep ranges are given explicitly rather than left at their defaults: with $\sim 10^{7}$ rotations in the
QPE ladder, each rotation must be synthesized far more accurately than the defaults allow, and the default
query returns no result that meets `max_error`.

In [ ]:
# this cell takes ~3 mins to run
from qdk.qre import LatticeSurgery, PSSPC, estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

MAJORANA_ERROR_RATE = 1e-5
ARCHITECTURE = Majorana(error_rate=MAJORANA_ERROR_RATE)
MAX_ERROR = 0.5


def estimate_physical(circuit: Circuit, name: str):
    """Qubit/runtime Pareto frontier for the given circuit on the Majorana architecture.

    Tracer memory grows like 2**num_bits * num_divisions * n_terms, which is what
    the timeout guards against.
    """
    application = circuit.get_qre_application()
    trace_query = (
        application.q()
        * PSSPC.q(num_ts_per_rotation=list(range(20, 45, 2)))
        * LatticeSurgery.q(slow_down_factor=[1.0 * j for j in range(1, 20)])
    )
    isa_query = ThreeAux.q() * RoundBasedFactory.q(code_query=ThreeAux.q())
    with time_limit(STEP_TIMEOUT_SECONDS, f"estimate_physical({name})"):
        return estimate(application, ARCHITECTURE, isa_query, trace_query, max_error=MAX_ERROR, name=name)

estimates = estimate_physical(circuit, f"{EXAMPLE_NUM_SITES}x{EXAMPLE_NUM_SITES} lattice")
print(estimates)

## Sweeping the lattice sizes

For each lattice we build the qubit Hamiltonian, derive $(m, r)$ from its target precision, compile the QPE
circuit and estimate it. The table reports the derived parameters together with the fastest point on each
Pareto frontier.

Note how $m$ stays constant across the sweep while $r$ *falls* to 1: the benchmark's precision target grows
with the lattice ($\epsilon \propto N$) at the same rate as $\lambda$, so $2\lambda/\epsilon$ is
size-independent, while $t_0 \propto 1/N$ shrinks fast enough that a single Strang step suffices once
$L \gtrsim 8$.

In [ ]:
# This cell takes 2 hrs
tables = []
rows = []

for size in BENCHMARK_LATTICE_SIZES:
    operator = qubit_operator(size)
    parameters = qpe_parameters(operator, target_precision(size))
    initial_state = reference_state_prep(size * size, num_electrons(size))
    table = estimate_physical(qpe_circuit(operator, parameters, initial_state), f"{size}x{size}")
    tables.append(table)

    fastest = min(table, key=lambda entry: entry.runtime)
    rows.append({
        "L": size,
        "sites": size * size,
        "qubits": operator.num_qubits,
        "terms": len(operator.pauli_strings),
        "electrons": num_electrons(size),
        "lambda": parameters.one_norm,
        "alpha_2 / N": parameters.alpha2 / (size * size),
        "num_bits": parameters.num_bits,
        "num_divisions": parameters.num_divisions,
        "physical qubits": fastest.qubits,
        "runtime (h)": fastest.runtime / 3.6e12,
    })
    print(f"{size}x{size} lattice: {fastest.runtime / 3.6e12} h {fastest.qubits} physical qubits")

pd.DataFrame(rows).set_index("L")

In [ ]:
fig = plot_estimates(tables, runtime_unit="hours", figsize=(12, 7))
ax = fig.axes[0]
ax.set_title("2D Fermi-Hubbard ground-state QPE, Majorana architecture")
fig.set_layout_engine("constrained")
fig

## How far the sweep can go

The benchmark specifies $L$ up to 200. Two things decide how far it actually gets, neither of them the
resource estimation itself:

1. **Building the qubit Hamiltonian and the circuit.** The term count grows like $2.75\,N$, and the mapping,
   the commutator bound and the Q# payload all grow with it. `STEP_TIMEOUT_SECONDS` caps each step so a single
   oversized lattice cannot stall the sweep.
2. **Tracer memory**, which grows like $2^{m} \times r \times n_{\mathrm{terms}}$. Two choices keep this in
   check: iterative QPE compiles a *single* controlled unitary instead of the whole $m$-rung ladder, and the
   budget planner above picks the $m$ that minimizes $2^{m} r$. This is also why the loose
   `trotter_steps_naive` bound is unusable here -- ignoring commutation it returns $r = 253$ at *every* size
   (since $\lambda \propto N$ cancels against $t_0 = \pi/\lambda$), roughly $100\times$ the commutator bound,
   and the tracer then attempts a 206 GB allocation at $L = 6$.

Qubitization-based alternatives such as FOQCS or SOSSA would replace the product formula in
`hamiltonian_unitary_builder` and are worth revisiting for the same benchmark:

- [2601.18767v1] Practical block encodings of matrix polynomials that can also be trivially controlled
- [2602.05069v1] Near-frustration-free electronic structure Hamiltonian representations and lower bound certificates

In [ ]:
import matplotlib.pyplot as plt

# For a periodic square lattice, there are 2N edges and two spin sectors.
# The hopping terms contribute 4|T|N to lambda and the on-site terms |U|N.
scaling_sizes = np.arange(2, 201)
scaling_sites = scaling_sizes**2
scaling_one_norm = (4 * abs(HOPPING_T) + abs(COULOMB_U)) * scaling_sites
scaling_precision = TARGET_PRECISION_PER_SITE * scaling_sites

# Reuse the planner so the analytic sweep and the executed lattices cannot drift apart.
scaling_plans = [
    plan_qpe(one_norm, precision, ALPHA2_PER_SITE * sites)
    for one_norm, precision, sites in zip(scaling_one_norm, scaling_precision, scaling_sites)
]

parameter_scaling = pd.DataFrame(
    {
        "one_norm": scaling_one_norm,
        "evolution_time": [plan.evolution_time for plan in scaling_plans],
        "num_bits": [plan.num_bits for plan in scaling_plans],
        "num_divisions": [plan.num_divisions for plan in scaling_plans],
    },
    index=pd.Index(scaling_sizes, name="L"),
)

# Keep the analytic sweep tied to the explicitly constructed example.
assert np.isclose(parameter_scaling.loc[EXAMPLE_NUM_SITES, "one_norm"], parameters.one_norm)
assert np.isclose(parameter_scaling.loc[EXAMPLE_NUM_SITES, "evolution_time"], parameters.evolution_time)
assert parameter_scaling.loc[EXAMPLE_NUM_SITES, "num_bits"] == parameters.num_bits

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
series = [
    ("one_norm", r"Coefficient 1-norm $\lambda$", "log"),
    ("evolution_time", r"Evolution time $t_0$", "log"),
    ("num_bits", "QPE phase bits", "linear"),
]
for ax, (column, title, yscale) in zip(axes, series):
    ax.plot(parameter_scaling.index, parameter_scaling[column], linewidth=2)
    value = parameter_scaling.loc[EXAMPLE_NUM_SITES, column]
    ax.scatter([EXAMPLE_NUM_SITES], [value], color="tab:red", zorder=3)
    ax.annotate(
        f"L={EXAMPLE_NUM_SITES}\n{value:.6g}",
        (EXAMPLE_NUM_SITES, value),
        xytext=(8, 8),
        textcoords="offset points",
    )
    ax.set(title=title, xlabel="Linear lattice size L", yscale=yscale)
    ax.grid(True, which="both", alpha=0.3)

fig.suptitle("2D Fermi-Hubbard QPE parameter scaling")
fig.set_layout_engine("constrained")
fig